# FinGraph AI — prvi korak u Google Colabu
Ovaj notebook **ne sadrži raw podatke**. Preuzmi ih od autora i stavi u `data/raw/`.
Prije pokretanja odaberi Runtime → Change runtime type → T4 GPU (ako je dostupan).

In [6]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/fingraph-ai/FinGraph_AI_MPVI_Starter

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/fingraph-ai/FinGraph_AI_MPVI_Starter


In [8]:
%cd /content/drive/MyDrive/fingraph-ai/FinGraph_AI_MPVI_Starter

%pip install -e .

/content/drive/MyDrive/fingraph-ai/FinGraph_AI_MPVI_Starter
Obtaining file:///content/drive/MyDrive/fingraph-ai/FinGraph_AI_MPVI_Starter
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fingraph-ai-mpvi (pyproject.toml) ... done
  Created wheel for fingraph-ai-mpvi: filename=fingraph_ai_mpvi-0.1.0-0.editable-py3-none-any.whl size=3953 sha256=e1712edb6699d1984d941dd3213d2d082015d7538eb85f8a35c5ca76e0f8600c
  Stored in directory: /tmp/pip-ephem-wheel-cache-hyzwll35/wheels/4a/5d/60/79403999ae9ede896fcf34eb1ef0c7792f8fe634893fdc644f
Successfully built fingraph-ai-mpvi


In [9]:
!python -m pip show fingraph-ai-mpvi
!python -c "import fingraph; print(fingraph.__file__)"

Name: fingraph-ai-mpvi
Version: 0.1.0
Summary: Reproducible financial graph anomaly detection experiments on Elliptic++.
Home-page: 
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Editable project location: /content/drive/MyDrive/fingraph-ai/FinGraph_AI_MPVI_Starter
Requires: matplotlib, numpy, pandas, scikit-learn, xgboost
Required-by: 
/content/drive/MyDrive/fingraph-ai/FinGraph_AI_MPVI_Starter/src/fingraph/__init__.py


In [11]:
import sys
import torch

project_path = "/content/drive/MyDrive/fingraph-ai/FinGraph_AI_MPVI_Starter"

sys.path.insert(0, f"{project_path}/src")

import fingraph

print("FinGraph AI uspješno učitan!")
print("PyTorch:", torch.__version__)
print("GPU dostupan:", torch.cuda.is_available())


FinGraph AI uspješno učitan!
PyTorch: 2.11.0+cu130
GPU dostupan: True


## 1. Provjeri dataset
Audit mora potvrditi strukturu prije treniranja. Ako su CSV-ovi samo Git LFS pokazivači, preuzmi prave datoteke s Google Drivea autora.

In [12]:
!python -m fingraph.audit --data-dir data/raw

{
  "nodes": 203769,
  "directed_edges": 234355,
  "features": 182,
  "missing_feature_values": 16405,
  "time_steps": 49,
  "labels": {
    "illicit": 4545,
    "licit": 42019,
    "unknown": 157205
  },
  "class_distribution_by_time": {
    "1": {
      "illicit": 17,
      "licit": 2130,
      "unknown": 5733
    },
    "2": {
      "illicit": 18,
      "licit": 1099,
      "unknown": 3427
    },
    "3": {
      "illicit": 11,
      "licit": 1268,
      "unknown": 5342
    },
    "4": {
      "illicit": 30,
      "licit": 1410,
      "unknown": 4253
    },
    "5": {
      "illicit": 8,
      "licit": 1874,
      "unknown": 4921
    },
    "6": {
      "illicit": 5,
      "licit": 480,
      "unknown": 3843
    },
    "7": {
      "illicit": 102,
      "licit": 1101,
      "unknown": 4845
    },
    "8": {
      "illicit": 67,
      "licit": 1098,
      "unknown": 3292
    },
    "9": {
      "illicit": 248,
      "licit": 530,
      "unknown": 4218
    },
    "10": {
      "illici

## 2. Treniranje i evaluacija
Pokreni redom. Metod koristi trening 1–34, validaciju 35–41 i test 42–49.

In [13]:
!python -m fingraph.baseline --data-dir data/raw

{
  "model": "XGBoost",
  "seed": 42,
  "train_end": 34,
  "valid_end": 41,
  "validation": {
    "n": 7829,
    "positives": 675,
    "threshold": 0.7699999999999999,
    "pr_auc": 0.9341036204145822,
    "roc_auc": 0.9786336574203502,
    "precision": 0.9769736842105263,
    "recall": 0.88,
    "f1": 0.9259547934528449,
    "precision_at_100_or_less": 1.0,
    "recall_at_100_or_less": 0.14814814814814814
  },
  "test": {
    "n": 8841,
    "positives": 408,
    "threshold": 0.7699999999999999,
    "pr_auc": 0.5674848876650866,
    "roc_auc": 0.8717689376236679,
    "precision": 0.8441558441558441,
    "recall": 0.47794117647058826,
    "f1": 0.6103286384976526,
    "precision_at_100_or_less": 1.0,
    "recall_at_100_or_less": 0.24509803921568626
  }
}


In [15]:
%pip install -q torch_geometric

In [16]:
!python -m fingraph.gnn --data-dir data/raw --epochs 60 --hidden-dim 64

Epoch 001 | loss=0.7883 | val PR-AUC=0.1691
Epoch 002 | loss=0.6540 | val PR-AUC=0.2071
Epoch 003 | loss=0.5671 | val PR-AUC=0.2429
Epoch 004 | loss=0.5080 | val PR-AUC=0.2757
Epoch 005 | loss=0.4735 | val PR-AUC=0.3036
Epoch 006 | loss=0.4439 | val PR-AUC=0.3224
Epoch 007 | loss=0.4204 | val PR-AUC=0.3332
Epoch 008 | loss=0.3966 | val PR-AUC=0.3425
Epoch 009 | loss=0.3850 | val PR-AUC=0.3468
Epoch 010 | loss=0.3653 | val PR-AUC=0.3507
Epoch 011 | loss=0.3541 | val PR-AUC=0.3532
Epoch 012 | loss=0.3453 | val PR-AUC=0.3540
Epoch 013 | loss=0.3341 | val PR-AUC=0.3535
Epoch 014 | loss=0.3283 | val PR-AUC=0.3546
Epoch 015 | loss=0.3200 | val PR-AUC=0.3554
Epoch 016 | loss=0.3111 | val PR-AUC=0.3560
Epoch 017 | loss=0.3066 | val PR-AUC=0.3574
Epoch 018 | loss=0.3003 | val PR-AUC=0.3602
Epoch 019 | loss=0.2948 | val PR-AUC=0.3641
Epoch 020 | loss=0.2926 | val PR-AUC=0.3689
Epoch 021 | loss=0.2854 | val PR-AUC=0.3748
Epoch 022 | loss=0.2818 | val PR-AUC=0.3811
Epoch 023 | loss=0.2789 | val PR

## 3. Rezultati
Metrike su sačuvane u `outputs/xgboost/metrics.json` i `outputs/graphsage/metrics.json`. Ne interpretirati predikcije na neoznačenim čvorovima kao dokaz krivičnih aktivnosti.

In [17]:
import json
from pathlib import Path
for model in ('xgboost', 'graphsage'):
    path = Path('outputs') / model / 'metrics.json'
    if path.exists():
        result = json.loads(path.read_text())
        print(model, result['test'])

xgboost {'n': 8841, 'positives': 408, 'threshold': 0.7699999999999999, 'pr_auc': 0.5674848876650866, 'roc_auc': 0.8717689376236679, 'precision': 0.8441558441558441, 'recall': 0.47794117647058826, 'f1': 0.6103286384976526, 'precision_at_100_or_less': 1.0, 'recall_at_100_or_less': 0.24509803921568626}
graphsage {'n': 8841, 'positives': 408, 'threshold': 0.8799999999999999, 'pr_auc': 0.3890913562522102, 'roc_auc': 0.7875924530846372, 'precision': 0.44410876132930516, 'recall': 0.3602941176470588, 'f1': 0.39783491204330174, 'precision_at_100_or_less': 0.85, 'recall_at_100_or_less': 0.20833333333333334}
